# Lesson 8: Multi-Step AI Workflows

## Chaining AI Calls Together

In this lesson, you'll learn:
- Why one AI call isn't always enough
- How to chain multiple AI calls together
- Passing output from one call as input to the next
- Build a Blog Post Generator (outline → sections → full post)!

**Complex tasks = Multiple AI calls working together!**


---

## Part 1: Why Chain AI Calls?

### One Call vs Multiple Calls

```
┌─────────────────────────────────────────────────────────────────┐
│ SINGLE CALL APPROACH (Limited)                                  │
├─────────────────────────────────────────────────────────────────┤
│ "Write me a complete blog post about Python"                    │
│                                                                 │
│ Problems:                                                       │
│ • AI might ramble or lose focus                                 │
│ • Less control over structure                                   │
│ • Harder to refine specific parts                               │
│ • May hit token limits for long content                         │
└─────────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────────┐
│ MULTI-STEP APPROACH (Better!)                                   │
├─────────────────────────────────────────────────────────────────┤
│ Step 1: "Create an outline for a blog post about Python"        │
│         → Get outline                                           │
│                                                                 │
│ Step 2: "Write the introduction based on this outline: {outline}"│
│         → Get introduction                                      │
│                                                                 │
│ Step 3: "Write section 1 about: {section_1_topic}"              │
│         → Get section 1                                         │
│                                                                 │
│ Step 4: Combine all parts into final post                       │
│                                                                 │
│ Benefits:                                                       │
│ • More control at each step                                     │
│ • Can review/modify between steps                               │
│ • Better quality output                                         │
│ • Works for any length                                          │
└─────────────────────────────────────────────────────────────────┘
```


---

## Part 2: Setup


In [ ]:
# Setup - Run this first!

from dotenv import load_dotenv
from openai import OpenAI
import json

load_dotenv(override=True)
client = OpenAI()

def chat(prompt, temperature=0.7):
    """Basic chat function"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=temperature
    )
    return response.choices[0].message.content

def get_json(prompt):
    """Get JSON response"""
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
        response_format={"type": "json_object"}
    )
    return json.loads(response.choices[0].message.content)

print("✅ Ready to learn multi-step workflows!")


---

## Part 3: Simple Chaining Example

Let's start with a simple example: generating a product name, then a tagline, then a description.


In [ ]:
# Simple 3-step chain

product_type = "smart water bottle"

print("🔗 MULTI-STEP PRODUCT CREATION")
print("=" * 50)

# Step 1: Generate a product name
print("\n📍 Step 1: Generating product name...")
name_prompt = f"Invent a creative brand name for a {product_type}. Reply with just the name."
product_name = chat(name_prompt, temperature=0.8).strip()
print(f"   Product Name: {product_name}")

# Step 2: Generate a tagline using the name
print("\n📍 Step 2: Creating tagline...")
tagline_prompt = f"Create a catchy 5-7 word tagline for '{product_name}', a {product_type}. Reply with just the tagline."
tagline = chat(tagline_prompt, temperature=0.8).strip()
print(f"   Tagline: {tagline}")

# Step 3: Generate a description using name and tagline
print("\n📍 Step 3: Writing description...")
desc_prompt = f"""Write a 2-sentence product description for:
Product: {product_name}
Type: {product_type}
Tagline: {tagline}

Make it exciting and highlight key benefits."""
description = chat(desc_prompt, temperature=0.7)
print(f"   Description: {description}")

# Final result
print("\n" + "=" * 50)
print("🎉 FINAL PRODUCT MARKETING:")
print(f"\n   {product_name}")
print(f"   \"{tagline}\"")
print(f"\n   {description}")


### Notice How Each Step Builds on the Previous!

- Step 1: Created a name → `product_name`
- Step 2: Used `product_name` to create a tagline → `tagline`
- Step 3: Used both to create a description

This is the core pattern of multi-step workflows!


---

## Part 4: Project - Blog Post Generator

Let's build a complete blog post generator that:
1. Creates an outline
2. Writes each section
3. Combines everything into a final post


In [ ]:
# PROJECT: Blog Post Generator

def generate_blog_post(topic, num_sections=3):
    """
    Generate a complete blog post using multi-step AI workflow.
    
    Args:
        topic: What the blog post is about
        num_sections: Number of main sections
    
    Returns:
        Complete blog post as a string
    """
    
    print(f"📝 GENERATING BLOG POST: {topic}")
    print("=" * 60)
    
    # STEP 1: Generate the outline
    print("\n📍 Step 1: Creating outline...")
    outline_prompt = f"""
    Create a blog post outline for: "{topic}"
    
    Return JSON with:
    {{
        "title": "catchy blog title",
        "introduction_hook": "one sentence to hook readers",
        "sections": [
            {{"heading": "section title", "key_points": ["point1", "point2"]}}
        ],
        "conclusion_theme": "main takeaway"
    }}
    
    Include exactly {num_sections} sections.
    """
    outline = get_json(outline_prompt)
    print(f"   Title: {outline['title']}")
    print(f"   Sections: {len(outline['sections'])}")
    
    # STEP 2: Write the introduction
    print("\n📍 Step 2: Writing introduction...")
    intro_prompt = f"""
    Write an engaging introduction (2-3 paragraphs) for a blog post.
    
    Title: {outline['title']}
    Hook: {outline['introduction_hook']}
    
    The introduction should grab attention and preview what's coming.
    """
    introduction = chat(intro_prompt)
    print("   ✓ Introduction complete")
    
    # STEP 3: Write each section
    sections_content = []
    for i, section in enumerate(outline['sections']):
        print(f"\n📍 Step 3.{i+1}: Writing '{section['heading']}'...")
        section_prompt = f"""
        Write a blog section (2-3 paragraphs) with this heading: {section['heading']}
        
        Key points to cover:
        {json.dumps(section['key_points'])}
        
        Make it informative and engaging. Don't include the heading in your response.
        """
        section_content = chat(section_prompt)
        sections_content.append({
            "heading": section['heading'],
            "content": section_content
        })
        print(f"   ✓ Section complete")
    
    # STEP 4: Write the conclusion
    print("\n📍 Step 4: Writing conclusion...")
    conclusion_prompt = f"""
    Write a strong conclusion (1-2 paragraphs) for a blog post.
    
    Title: {outline['title']}
    Main takeaway: {outline['conclusion_theme']}
    
    Summarize key points and end with a call to action.
    """
    conclusion = chat(conclusion_prompt)
    print("   ✓ Conclusion complete")
    
    # STEP 5: Combine everything
    print("\n📍 Step 5: Assembling final post...")
    
    full_post = f"# {outline['title']}\n\n"
    full_post += introduction + "\n\n"
    
    for section in sections_content:
        full_post += f"## {section['heading']}\n\n"
        full_post += section['content'] + "\n\n"
    
    full_post += "## Conclusion\n\n"
    full_post += conclusion
    
    print("   ✓ Blog post complete!")
    
    return full_post

print("✅ Blog Post Generator ready!")


In [ ]:
# Generate a blog post!

blog_post = generate_blog_post(
    topic="Why Python is Perfect for Beginners",
    num_sections=3
)


In [ ]:
# Display the final blog post

from IPython.display import Markdown, display

print("\n" + "=" * 60)
print("📄 FINAL BLOG POST")
print("=" * 60 + "\n")

display(Markdown(blog_post))


---

## Summary: Key Takeaways

### Multi-Step Workflow Pattern

```python
# Step 1: Get initial data
result_1 = chat("First task...")

# Step 2: Use result_1 to get result_2
result_2 = chat(f"Second task using {result_1}...")

# Step 3: Use result_2 to get result_3
result_3 = chat(f"Third task using {result_2}...")

# Combine results
final_output = combine(result_1, result_2, result_3)
```

### When to Use Multi-Step

| Use Multi-Step When | Stay Single When |
|--------------------|--------------------|
| Complex, long content | Simple questions |
| Need structured output | Quick responses |
| Want control over parts | One-shot tasks |
| Building on previous results | Independent requests |

### What You Built
A **Blog Post Generator** that creates professional posts through a 5-step workflow!

### What's Next?
In **Lesson 9**, you'll learn about **AI Agents** - AI that can make decisions and take actions!

**Amazing progress! Move on to Lesson 9!**
